In [1]:
"Hello World"

'Hello World'

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import urllib3
import html5lib

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


url = 'https://josaa.admissions.nic.in/applicant/seatmatrix/openingclosingrankarchieve.aspx'

params = {
    "ctl00$ContentPlaceHolder1$ddlInstype": "ALL",
    "ctl00$ContentPlaceHolder1$ddlInstitute": "ALL",
    "ctl00$ContentPlaceHolder1$ddlBranch": "ALL",
    "ctl00$ContentPlaceHolder1$ddlSeatType": "ALL",
    "ctl00$ContentPlaceHolder1$btnSubmit": "Submit"
}


years = [
    "2018",
    "2017"
#     "2017",
#     "2016"
]

rounds = [
    "1",
    "2",
    "3",
    "4",
    "5",
#     "6"
    "6",
    "7"
]



def josaa_scrape(year, Round):
    """
    Sample usage: df = josaa_scrape("2018", "1")
    df.info()
    """
    with requests.Session() as s:
        s.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'})
        R = s.get(url, verify=False)
        data = {}
        data.update({tag['name']: tag['value'] for tag in BeautifulSoup(R.content, 'lxml').select('input[name^=__]')})
        data["ctl00$ContentPlaceHolder1$ddlYear"] = year
        R = s.post(url, data=data, verify=False)

        data.update({tag['name']: tag['value'] for tag in BeautifulSoup(R.content, 'lxml').select('input[name^=__]')})
        data["ctl00$ContentPlaceHolder1$ddlroundno"] = Round
        R = s.post(url, data=data, verify=False)

        for key, value in params.items():
            data.update({tag['name']: tag['value'] for tag in BeautifulSoup(R.content, 'lxml').select('input[name^=__]')})
            data[key] = value
            R = s.post(url, data=data, verify=False)

    import io
    table = BeautifulSoup(R.text, 'lxml').find(id = 'ctl00_ContentPlaceHolder1_GridView1')
    df = pd.read_html(io.StringIO(str(table)))[0]
    df.dropna(inplace = True, how="all")

    df["Year"] = year
    df["Round"] = Round
    df['Opening Rank'] = df['Opening Rank'].astype(str).str.extract(r'(\d+)')[0].astype(float)
    df['Closing Rank'] = df['Closing Rank'].astype(str).str.extract(r'(\d+)')[0].astype(float)

    return df





In [6]:
years = [
    "2016",
    "2017",
    "2018",
    "2019",
    "2020",
    "2021",
    "2022",
    "2023",
    "2024",
    "2025"
]

rounds = [
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7"
]

import os

os.makedirs("data", exist_ok=True)
for year in  years:
    for Round in rounds:

        print(f"Scraping Year: {year}, Round: {Round}")
        try:
            df = josaa_scrape(year, Round)
            df.to_csv(f"data/josaa_scrape_{year}_round_{Round}.csv", index=False)
        except Exception as e:
            print(f"Error scraping Year: {year}, Round: {Round} - {e}")



Scraping Year: 2016, Round: 1
Scraping Year: 2016, Round: 2
Scraping Year: 2016, Round: 3
Scraping Year: 2016, Round: 4
Scraping Year: 2016, Round: 5
Scraping Year: 2016, Round: 6
Scraping Year: 2016, Round: 7
Error scraping Year: 2016, Round: 7 - `Import html5lib` failed.  Use pip or conda to install the html5lib package.
Scraping Year: 2017, Round: 1
Scraping Year: 2017, Round: 2
Scraping Year: 2017, Round: 3
Scraping Year: 2017, Round: 4
Scraping Year: 2017, Round: 5
Scraping Year: 2017, Round: 6
Scraping Year: 2017, Round: 7
Scraping Year: 2018, Round: 1
Scraping Year: 2018, Round: 2
Scraping Year: 2018, Round: 3
Scraping Year: 2018, Round: 4
Scraping Year: 2018, Round: 5
Scraping Year: 2018, Round: 6
Scraping Year: 2018, Round: 7
Scraping Year: 2019, Round: 1
Scraping Year: 2019, Round: 2
Scraping Year: 2019, Round: 3
Scraping Year: 2019, Round: 4
Scraping Year: 2019, Round: 5
Scraping Year: 2019, Round: 6
Scraping Year: 2019, Round: 7
Scraping Year: 2020, Round: 1
Scraping Year: 

In [17]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import urllib3
import html5lib

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


# Current ORCR endpoint (newer years) for JOSAA
url = 'https://josaa.admissions.nic.in/applicant/SeatAllotmentResult/CurrentORCR.aspx'

params = {
    "ctl00$ContentPlaceHolder1$ddlInstype": "ALL",
    "ctl00$ContentPlaceHolder1$ddlInstitute": "ALL",
    "ctl00$ContentPlaceHolder1$ddlBranch": "ALL",
    "ctl00$ContentPlaceHolder1$ddlSeattype": "ALL",
    "ctl00$ContentPlaceHolder1$btnSubmit": "Submit"
}


def josaa_scrape(year, Round):
    """
    Sample usage: df = josaa_scrape("2018", "1")
    df.info()
    """
    with requests.Session() as s:
        s.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'})

        # initial page load
        R = s.get(url, verify=False)
        soup = BeautifulSoup(R.content, 'lxml')

        data = {tag['name']: tag.get('value', '') for tag in soup.select('input[name^=__]') if tag.has_attr('name')}

        # Year may not exist for current endpoint; apply if present
        year_selector = soup.select_one('[name="ctl00$ContentPlaceHolder1$ddlYear"]')
        if year_selector is not None:
            data["ctl00$ContentPlaceHolder1$ddlYear"] = year
            data["__EVENTTARGET"] = "ctl00$ContentPlaceHolder1$ddlYear"
            R = s.post(url, data=data, verify=False)
            soup = BeautifulSoup(R.content, 'lxml')
            data.update({tag['name']: tag.get('value', '') for tag in soup.select('input[name^=__]') if tag.has_attr('name')})

        # Round selection
        round_selector = soup.select_one('[name="ctl00$ContentPlaceHolder1$ddlroundno"]')
        if round_selector is not None:
            data["ctl00$ContentPlaceHolder1$ddlroundno"] = Round
            data["__EVENTTARGET"] = "ctl00$ContentPlaceHolder1$ddlroundno"
            R = s.post(url, data=data, verify=False)

        # now apply filters and submit
        for key, value in params.items():
            soup = BeautifulSoup(R.content, 'lxml')
            data.update({tag['name']: tag.get('value', '') for tag in soup.select('input[name^=__]') if tag.has_attr('name')})
            data[key] = value
            
            if key == "ctl00$ContentPlaceHolder1$btnSubmit":
                data.pop('__EVENTTARGET', None)
                data.pop('__EVENTARGUMENT', None)
            else:
                data['__EVENTTARGET'] = key
                data['__EVENTARGUMENT'] = ''
                
            R = s.post(url, data=data, verify=False)

    # Parse result table
    import io
    soup = BeautifulSoup(R.text, 'lxml')
    table = soup.find(id='ctl00_ContentPlaceHolder1_GridView1')

    print(f"Response status: {R.status_code}")
    print(f"Table found: {table is not None}")
    if not table:
        raise ValueError('Unable to locate result table. Page may have changed or year/round not available.')

    df = pd.read_html(io.StringIO(str(table)), flavor='lxml')[0]
    if df.empty:
        raise ValueError('Parsed DataFrame is empty; check whether data exists for this year/round.')

    df.dropna(inplace=True, how='all')
    df["Year"] = year
    df["Round"] = Round

    # convert ranks to numeric if possible
    for col in ['Opening Rank', 'Closing Rank']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.extract(r'(\d+)')[0]
            df[col] = pd.to_numeric(df[col], errors='coerce')

    return df



rounds = [
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7"
]

import os

# script_dir = os.path.dirname(os.path.abspath(__file__))
# data_dir = os.path.join(script_dir, "data")
os.makedirs("data", exist_ok=True)

for Round in rounds:
    try:
        print(f"Scraping Year: {2025}, Round: {Round}")
        df = josaa_scrape(2025, Round)
        df.to_csv(os.path.join("data", f"josaa_scrape_{2025}_round_{Round}.csv"), index=False)
    except Exception as e:
        print(f"Error scraping Year: {2025}, Round: {Round} - {e}")


Scraping Year: 2025, Round: 1
Response status: 200
Table found: True
Scraping Year: 2025, Round: 2
Response status: 200
Table found: True
Scraping Year: 2025, Round: 3
Response status: 200
Table found: True
Scraping Year: 2025, Round: 4
Response status: 200
Table found: True
Scraping Year: 2025, Round: 5
Response status: 200
Table found: True
Scraping Year: 2025, Round: 6
Response status: 200
Table found: True
Scraping Year: 2025, Round: 7
Response status: 200
Table found: False
Error scraping Year: 2025, Round: 7 - Unable to locate result table. Page may have changed or year/round not available.
